# Conditioned variational autoencoder (conditioned VAE) for icon generation
### Add a word condition to the same VAE idea, with explicit word control

We already know that CNNs are good at learning **spatial patterns** in images. Both previous lessons also use CNNs; here we add the word condition to the encoder and decoder.

In the conditioned VAE:

```text
IMAGE + WORD → CNN ENCODER → latent z → CNN DECODER + WORD → IMAGE
```

Five words: **heart · face · robot · tree · spaceship**


## 1. Download the pre-generated training dataset

We download **10,000 fixed 16×16 black-and-white icons** from GitHub: 2,000 examples per word.
The images have already been generated; Colab only downloads and loads them.

This version uses the improved icon templates, including a correctly shaped heart.


In [ ]:
# Download the pre-generated training data. No image generator runs here.
from pathlib import Path
import hashlib
import urllib.request

DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-black-white-generative-ai/main/data/icons_cnn_v2.npz'
DATA_FILE = Path('icons_cnn_v2.npz')
DATA_SHA256 = 'f5efcac36774ebced04128dc85d297cca9fc11b5469b2baa9be459c00b834006'

def checksum_matches(path):
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == DATA_SHA256

# Reuse the local file when this cell is run again in the same Colab session.
if not checksum_matches(DATA_FILE):
    temporary = DATA_FILE.with_suffix('.download')
    urllib.request.urlretrieve(DATA_URL, temporary)
    if not checksum_matches(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError('Dataset checksum mismatch. Please rerun this cell.')
    temporary.replace(DATA_FILE)

print('Training dataset ready:', DATA_FILE, '(10,000 pre-generated icons)')


## 2. Load the black-and-white images

Black = 0. White = 1 after normalization. Labels identify the five condition words.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducible classroom training (exact results can still vary by hardware).
keras.utils.set_random_seed(42)

# NPZ is a compressed collection of NumPy arrays, already prepared on GitHub.
# allow_pickle=False keeps loading limited to ordinary numeric/string arrays.
with np.load(DATA_FILE, allow_pickle=False) as data:
    X = data['images'].astype(np.float32) / 255.0
    y = data['labels'].astype(np.int32)
    words = data['words'].tolist()

# Check image dimensions and the word-to-label mapping before training.
assert X.shape == (10000, 16, 16)
assert y.shape == (10000,)
assert words == ['heart', 'face', 'robot', 'tree', 'spaceship']
assert np.isin(X, [0.0, 1.0]).all()
assert np.array_equal(np.bincount(y, minlength=5), [2000] * 5)

# CNNs expect a channel dimension: (10000, 16, 16, 1).
X = np.expand_dims(X, axis=-1)

print('Images:', X.shape)
print('Labels:', y.shape)
print('Condition words:', words)


### Look at one example of each object


In [ ]:
plt.figure(figsize=(10,2))
for i, word in enumerate(words):
    index = np.where(y == i)[0][0]
    plt.subplot(1,5,i+1)
    plt.imshow(X[index,:,:,0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    plt.title(word)
    plt.axis('off')
plt.show()


# 3. CNN Encoder

The encoder's job is to **compress** an image.

```text
16×16 image
    ↓ Conv2D
 8×8×32
    ↓ Conv2D
 4×4×64
    ↓
 latent z
```

The word embedding tells the encoder which object the image represents.


In [ ]:
LATENT_DIM = 8
WORD_DIM = 8

# ----- TWO INPUTS -----
image_input = keras.Input(shape=(16,16,1), name='image')
word_input = keras.Input(shape=(), dtype='int32', name='word')

# ----- CNN IMAGE PATH -----
# stride=2 cuts width and height in half: 16x16 -> 8x8.
x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(image_input)

# Another stride=2: 8x8 -> 4x4.
x = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)

# Convert the 4x4x64 feature map into one vector.
x = layers.Flatten()(x)

# ----- WORD PATH -----
# Turn heart/face/robot/tree/spaceship into a learned vector.
word_vector = layers.Embedding(input_dim=5, output_dim=WORD_DIM)(word_input)
word_vector = layers.Flatten()(word_vector)

# Combine visual features with the word representation.
x = layers.Concatenate()([x, word_vector])
x = layers.Dense(128, activation='relu')(x)

# A VAE learns a DISTRIBUTION in latent space, not one exact point.
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)

class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        noise = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * noise

z = Sampling()([z_mean, z_log_var])

encoder = keras.Model([image_input, word_input], [z_mean, z_log_var, z], name='CNN_encoder')
encoder.summary()


# 4. CNN Decoder / Generator

Now we go in the opposite direction:

```text
z + word
   ↓ Dense
 4×4×64
   ↓ Conv2DTranspose
 8×8×64
   ↓ Conv2DTranspose
16×16×32
   ↓
16×16 image
```

A useful mental model: **the encoder shrinks; the decoder grows.**


In [ ]:
z_input = keras.Input(shape=(LATENT_DIM,), name='z')
decoder_word_input = keras.Input(shape=(), dtype='int32', name='word')

# Convert the requested word into numbers.
decoder_word_vector = layers.Embedding(input_dim=5, output_dim=WORD_DIM)(decoder_word_input)
decoder_word_vector = layers.Flatten()(decoder_word_vector)

# WHAT to draw (word) + HOW it varies (z).
x = layers.Concatenate()([z_input, decoder_word_vector])

# Create enough numbers for a 4x4x64 feature map.
x = layers.Dense(4*4*64, activation='relu')(x)
x = layers.Reshape((4,4,64))(x)

# Grow 4x4 -> 8x8.
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)

# Grow 8x8 -> 16x16.
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)

# Produce ONE black/white probability for every pixel.
# sigmoid keeps each output between 0 and 1.
output_image = layers.Conv2D(1, 3, padding='same', activation='sigmoid')(x)

decoder = keras.Model([z_input, decoder_word_input], output_image, name='CNN_decoder')
decoder.summary()


# 5. The two VAE losses

A VAE has two goals:

### Reconstruction loss
**“Can I draw the training image correctly?”**

Because every pixel is black or white, we use Keras's built-in **BinaryCrossentropy**.

### KL loss
**“Is the latent space organized so that random sampling works?”**

We hide its formula inside one small helper function. For this lesson, the important idea is its purpose, not memorizing the formula.


**Why KL needs enough weight:** training samples z from the encoder's image-specific distributions, while generation samples z from N(0, I). If KL is too weak, those distributions can drift apart: reconstruction looks good but new random samples look poor. We use beta = 1.0 with reconstruction summed over pixels.


In [ ]:
# Built-in Keras loss for black/white pixels.
reconstruction_loss_fn = keras.losses.BinaryCrossentropy(reduction='none')

# The special regularization term that makes this a VARIATIONAL autoencoder.
def kl_loss(z_mean, z_log_var):
    return -0.5 * tf.reduce_mean(
        tf.reduce_sum(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=1
        )
    )

# Reconstruction is SUMMED over 256 pixels, so use standard VAE KL weighting.
# Too little KL lets encoded points drift far from the N(0, I) used for generation.
BETA = 1.0


# 6. Train

Notice how short the important part now is:

```text
encode → decode → reconstruction loss + KL loss → learn
```


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = keras.metrics.Mean(name='loss')
        self.reconstruction_tracker = keras.metrics.Mean(name='reconstruction')
        self.kl_tracker = keras.metrics.Mean(name='KL')

    @property
    def metrics(self):
        # Keras resets these averages at the start of each epoch.
        return [self.loss_tracker, self.reconstruction_tracker, self.kl_tracker]

    def train_step(self, data):
        images, word_ids = data

        with tf.GradientTape() as tape:
            # STEP 1: Image -> latent z
            z_mean, z_log_var, z = self.encoder([images, word_ids], training=True)

            # STEP 2: latent z + word -> reconstructed image
            reconstructed = self.decoder([z, word_ids], training=True)

            # GOAL 1: Make the reconstructed pixels match the real pixels.
            pixel_loss = reconstruction_loss_fn(images, reconstructed)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(pixel_loss, axis=[1,2]))

            # GOAL 2: Keep latent space organized.
            latent_loss = kl_loss(z_mean, z_log_var)

            # Total VAE loss.
            total_loss = reconstruction_loss + BETA * latent_loss

        # Standard backpropagation — same idea as our previous neural networks.
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))

        # Report epoch averages instead of only the last batch's loss.
        batch_size = tf.cast(tf.shape(images)[0], tf.float32)
        self.loss_tracker.update_state(total_loss, sample_weight=batch_size)
        self.reconstruction_tracker.update_state(reconstruction_loss, sample_weight=batch_size)
        self.kl_tracker.update_state(latent_loss, sample_weight=batch_size)
        return {metric.name: metric.result() for metric in self.metrics}

dataset = tf.data.Dataset.from_tensor_slices((X,y))
dataset = dataset.shuffle(len(X), seed=42).batch(128).prefetch(tf.data.AUTOTUNE)

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(0.001))

# More data + CNN usually needs only a moderate number of epochs.
history = vae.fit(dataset, epochs=40)


# 7. Generate new images

Generation uses only the **decoder**:

```text
word + random z → decoder → new image
```


In [ ]:
word_to_id = {word:i for i,word in enumerate(words)}

def generate(word, number=10, binary=True, seed=None):
    if word not in word_to_id:
        print('Choose one of:', words)
        return

    # New random latent points = new variations.
    # A fixed seed reproduces the same z for fair soft/binary comparisons.
    random_z = np.random.default_rng(seed).normal(size=(number, LATENT_DIM)).astype('float32')
    word_ids = np.full(number, word_to_id[word], dtype=np.int32)

    generated = decoder.predict([random_z, word_ids], verbose=0)

    plt.figure(figsize=(15,2))
    for i in range(number):
        image = generated[i,:,:,0]

        # The neural network outputs probabilities between 0 and 1.
        # For pixel art, thresholding makes the final display truly black/white.
        if binary:
            image = image > 0.5

        plt.subplot(1,number,i+1)
        plt.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        plt.axis('off')
    plt.suptitle('Prompt: ' + word, fontsize=16)
    plt.show()

generate('robot')


## 8. Same word, different z

Run each line again. The **word stays fixed**, while random `z` creates variations.


In [ ]:
generate('heart')
generate('robot')
generate('spaceship')


## 9. See the raw probabilities vs. black/white pixels

The decoder actually predicts probabilities. Thresholding is only for display; **we never threshold during training**.


In [ ]:
# Soft probability image
generate('tree', number=8, binary=False, seed=42)

# Exactly the same idea, displayed as black/white pixel art
generate('tree', number=8, binary=True, seed=42)


## 10. Interactive generator


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

box = widgets.Dropdown(options=words, value='robot', description='Word:')
button = widgets.Button(description='Generate')
output = widgets.Output()

def clicked(_):
    with output:
        clear_output(wait=True)
        generate(box.value)

button.on_click(clicked)
display(box, button, output)


# AE → VAE → Conditioned VAE

| | AE | VAE | Conditioned VAE |
|---|---|---|---|
| Encoder | Image → z1, z2 | Image → mu, log-variance → sampled z | Image + word → mu, log-variance → sampled z |
| Decoder | z → image | z → image | z + word → image |
| Loss | Reconstruction | Reconstruction + KL | Reconstruction + KL |
| Latent dimensions | 2 | 2 | 8 |
| Can request an object by word? | No | No | Yes |

All three lessons use the same pre-generated 10,000 improved icons, CNN image paths, summed binary crossentropy, and 40 epochs. AE and VAE use two dimensions for visualization; the conditioned model retains eight for better icon detail and also adds word embeddings. This is a teaching progression, not a controlled performance benchmark.

The word provides object identity explicitly, while z can represent variation. This does **not** guarantee that z contains no class information: the encoder sees both the image and the word.
